In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from sqlalchemy import text

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

from cryptoquant.database.session import get_session
from cryptoquant.config.settings import get_settings

print("✓ Imports successful")
print(f"Current time: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC")

In [ ]:
session = get_session()
settings = get_settings()
print(f"✓ Connected to: {settings.database_url}")

In [ ]:
# Detect market_prices without technical_analysis (after warmup period)
query = text("""
WITH warmup_threshold AS (
    SELECT
        tp.id AS trading_pair_id,
        tp.symbol,
        MIN(mp.timestamp) AS first_candle,
        DATEADD(HOUR, 200, MIN(mp.timestamp)) AS warmup_complete
    FROM crypto.market_prices AS mp
    INNER JOIN crypto.trading_pairs AS tp ON tp.id = mp.trading_pair_id
    GROUP BY tp.id, tp.symbol
)
SELECT
    tp.symbol AS currency_pair,
    mp.id AS market_price_id,
    mp.timestamp,
    mp.close AS price
FROM crypto.market_prices AS mp
INNER JOIN crypto.trading_pairs AS tp ON tp.id = mp.trading_pair_id
INNER JOIN warmup_threshold AS w ON w.trading_pair_id = tp.id
LEFT JOIN crypto.technical_analysis AS ta ON ta.market_price_id = mp.id
WHERE mp.timestamp >= w.warmup_complete
  AND ta.id IS NULL
ORDER BY tp.symbol, mp.timestamp
""")

df_missing = pd.DataFrame(session.execute(query).fetchall(),
                          columns=['currency_pair', 'market_price_id', 'timestamp', 'price'])

print(f"=== Detected {len(df_missing)} Missing Analysis Record(s) ===")
if len(df_missing) > 0:
    display(df_missing.head(20))
    if len(df_missing) > 20:
        print(f"\n... and {len(df_missing) - 20} more records")
else:
    print("\n✓ No missing analysis detected!")

In [ ]:
# Summary by pair with date ranges
if len(df_missing) > 0:
    summary = df_missing.groupby('currency_pair').agg({
        'market_price_id': 'count',
        'timestamp': ['min', 'max']
    })
    summary.columns = ['missing_count', 'first_missing', 'last_missing']
    summary = summary.sort_values('missing_count', ascending=False)
    
    print("\n=== Missing Analysis Summary by Pair ===")
    display(summary)

In [ ]:
# Detect gaps in existing analysis (similar to candle gaps)
query = text("""
WITH ordered_analysis AS (
    SELECT
        ta.trading_pair_id,
        tp.symbol,
        ta.timestamp AS gap_start,
        LEAD(ta.timestamp) OVER (PARTITION BY ta.trading_pair_id ORDER BY ta.timestamp) AS gap_end,
        DATEDIFF(HOUR, ta.timestamp, LEAD(ta.timestamp) OVER (PARTITION BY ta.trading_pair_id ORDER BY ta.timestamp)) AS hours_gap
    FROM crypto.technical_analysis AS ta
    INNER JOIN crypto.trading_pairs AS tp ON tp.id = ta.trading_pair_id
)
SELECT
    symbol AS currency_pair,
    gap_start,
    gap_end,
    hours_gap
FROM ordered_analysis
WHERE gap_end IS NOT NULL
  AND hours_gap > 1
ORDER BY symbol, gap_start
""")

df_analysis_gaps = pd.DataFrame(session.execute(query).fetchall(),
                                columns=['currency_pair', 'gap_start', 'gap_end', 'hours_gap'])

print(f"\n=== Detected {len(df_analysis_gaps)} Analysis Gap(s) ===")
if len(df_analysis_gaps) > 0:
    display(df_analysis_gaps)
    total_hours = df_analysis_gaps['hours_gap'].sum()
    print(f"\n⚠️  Total missing analysis hours: {total_hours:,.0f}")
else:
    print("✓ No gaps in analysis sequence!")

In [ ]:
# Generate backfill commands
print("\n=== Backfill Commands ===\n")

if len(df_missing) > 0 or len(df_analysis_gaps) > 0:
    print("# For missing analysis records, run incremental mode:")
    print("python scripts/calculate_technical_analysis.py --mode incremental\n")
    
    if len(df_missing) > 0:
        for pair in df_missing['currency_pair'].unique():
            pair_data = df_missing[df_missing['currency_pair'] == pair]
            count = len(pair_data)
            start = pair_data['timestamp'].min()
            end = pair_data['timestamp'].max()
            print(f"# {pair}: {count} missing records from {start} to {end}")
            print(f"python scripts/calculate_technical_analysis.py --mode incremental --pair {pair}\n")
    
    if len(df_analysis_gaps) > 0:
        print("\n# For gaps in analysis, consider historical mode for specific periods:")
        for idx, row in df_analysis_gaps.iterrows():
            pair = row['currency_pair']
            start = row['gap_start']
            end = row['gap_end']
            hours = row['hours_gap']
            print(f"# {pair}: {hours} hour gap from {start} to {end}")
            # Calculate days needed
            days = max(1, int(hours / 24) + 1)
            print(f"python scripts/calculate_technical_analysis.py --mode historical --days {days} --pair {pair}\n")
else:
    print("✓ No backfill needed - all analysis is complete!")

In [ ]:
session.close()
print("✓ Database connection closed")